# Complete LangChain + Groq Notebook

## PGD Generative AI — Week 2 Practical



### Topics covered

1. Installing LangChain and Groq packages
2. Adding the Groq API key using Colab Secrets
3. Testing the installation
4. Basic Groq model invocation
5. Temperature and creativity
6. System and human messages
7. Multiple prompts
8. Streaming responses
9. Prompt templates
10. Local Hugging Face embeddings
11. Document embeddings
12. Cosine similarity and semantic search



## 1. Install the required packages

Run the following cell once. Colab installs these packages inside its temporary runtime.

The installation may take a few minutes because `sentence-transformers` downloads supporting libraries.


In [ ]:
%pip install -qU --no-cache-dir \
  torch torchvision torchaudio \
  sentence-transformers \
  langchain \
  langchain-core \
  langchain-groq \
  langchain-huggingface \
  scikit-learn \
  pandas \
  numpy

In [ ]:
import torch

print(torch.__version__)
print("Torch imported successfully")

2.13.0+cu130
Torch imported successfully


In [ ]:
import sentence_transformers

print("Sentence Transformers imported successfully")

Sentence Transformers imported successfully


## 2. Add the Groq API key securely

### Recommended method: Colab Secrets

1. Open the **key icon** in the left sidebar.
2. Click **Add new secret**.
3. Enter the name `GROQ_API_KEY`.
4. Paste your Groq API key as its value.
5. Enable **Notebook access**.

The next cell first checks Colab Secrets. If the secret is unavailable, it securely asks you to paste the key without displaying it.


In [ ]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_api_key = userdata.get("Groq_API")
except Exception:
    groq_api_key = None

if not groq_api_key:
    groq_api_key = getpass("Enter your GROQ_API_KEY: ")

os.environ["GROQ_API_KEY"] = groq_api_key

print("Groq API key configured successfully.")


Groq API key configured successfully.


## 3. Verify the installation

This cell confirms that the main packages are available.


In [ ]:
import numpy
import scipy
import sklearn
import sentence_transformers
import langchain_groq
import langchain_huggingface

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Sentence Transformers imported successfully")
print("LangChain Groq imported successfully")
print("LangChain Hugging Face imported successfully")

NumPy: 2.5.1
SciPy: 1.16.3
Scikit-learn: 1.9.0
Sentence Transformers imported successfully
LangChain Groq imported successfully
LangChain Hugging Face imported successfully


# Part A — Groq Chat Models

We will use the fast Groq-hosted model:

```text
llama-3.1-8b-instant
```

The same basic pattern is used throughout the notebook:

```python
model = ChatGroq(...)
result = model.invoke("Your prompt")
print(result.content)
```


## 4. Create a reusable Groq model

A temperature of `0` is suitable for factual and consistent responses.


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

print("Groq model is ready.")

Groq model is ready.


## 5. Basic LLM example



In [ ]:
result = llm.invoke("What is the capital of Italy?")

print(result.content)

The capital of Italy is Rome.


## 6. Basic chat-model example

Ask the model to create a short poem.


In [ ]:
creative_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=1.2,
)

result = creative_model.invoke("Write a 12-line poem about GTA 6.")

print(result.content)

In San Andreas' sun-kissed night so bright,
A new era dawns, a revolution in sight.
Grand Theft Auto 6, the wait is almost gone,
A new world awaits, with story yet unknown.

The streets of Vice City call again,
This time with twists, and a new game plan to win.
Players will return, with characters anew,
In a gripping tale of crime, and adventures true.

A new generation game, with a fresh take bold,
GTA 6 brings the heat, with a story to be told.
Will it top its predecessors, or bring something new?
Only time will tell, when this game is revealed to you.


## 7. Understanding temperature

Temperature controls variation in the generated output.

- `0.0`: focused and consistent
- `0.7`: balanced
- `1.5`: more creative and varied

Run this cell more than once to observe the difference.


In [ ]:
prompt = "Suggest three creative names for an AI education platform."

temperature_values = [0.0, 0.7, 1.5]

for value in temperature_values:
    model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=value,
    )

    result = model.invoke(prompt)

    print("=" * 100)
    print(f"Temperature: {value}")
    print(result.content)
    print()

Temperature: 0.0
Here are three creative name suggestions for an AI education platform:

1. **Lumin**: This name suggests illumination, knowledge, and understanding, which are all key aspects of education. It also has a modern and futuristic feel to it, which aligns well with the concept of AI.

2. **MindSpark**: This name conveys the idea of sparking creativity, curiosity, and innovation in learners. It also implies a dynamic and interactive learning experience, which is often associated with AI-powered education platforms.

3. **NexaLearn**: This name combines the words "nexus" and "learn," suggesting a connection or hub for learning. It also has a sleek and modern sound to it, which could appeal to a wide range of users. The "Nexa" prefix also implies a sense of cutting-edge technology, which is fitting for an AI education platform.

These names are just suggestions, and you may want to consider factors such as brand identity, target audience, and competition when choosing a name fo

### Student activity

Change the prompt below and compare the results at different temperatures.

Suggested prompts:

- Generate names for a healthcare chatbot.
- Write a slogan for an education institute.
- Suggest ideas for a business campaign.


In [ ]:
student_prompt = "Suggest three creative names for a healthcare chatbot."

for value in [0.0, 1.0, 1.8]:
    model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=value,
    )

    result = model.invoke(student_prompt)

    print(f"\nTemperature {value}")
    print("-" * 40)
    print(result.content)


Temperature 0.0
----------------------------------------
Here are three creative name suggestions for a healthcare chatbot:

1. **MedMind**: This name suggests a chatbot that is intelligent and insightful, providing users with helpful medical information and guidance. It also has a friendly and approachable tone, which can help put users at ease.

2. **DocChat**: This name incorporates the idea of a doctor or medical professional, while also emphasizing the chatbot's conversational nature. It's simple and easy to remember, making it a great choice for a healthcare chatbot.

3. **HealthHive**: This name positions the chatbot as a central hub or resource for users' health-related questions and concerns. It also has a friendly, community-oriented feel, which can help users feel more connected to the chatbot and more likely to engage with it.

I hope these suggestions are helpful! Let me know if you have any other questions.

Temperature 1.0
----------------------------------------
Here a

## 8. System and Human Messages

A **system message** defines the model's role, behaviour, tone, or rules.

A **human message** contains the user's request.


In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

teacher_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

messages = [
    SystemMessage(
        content="You are a helpful PGD teacher. Explain concepts in easy words and give one practical example."
    ),
    HumanMessage(
        content="What is the recipe to make Biryani?"
    ),
]

result = teacher_model.invoke(messages)

print(result.content)

Biryani is a popular South Asian dish made with a mixture of spices, basmati rice, and marinated meat or vegetables. Here's a simple recipe to make Biryani:

**Ingredients:**

For the rice:
- 2 cups basmati rice
- 4 cups water
- 1 tablespoon ghee or oil
- Salt to taste

For the marinade:
- 1 pound boneless chicken or beef (or vegetables like carrots, peas, and cauliflower)
- 1/2 cup yogurt
- 2 tablespoons lemon juice
- 1 teaspoon garam masala powder
- 1 teaspoon cumin powder
- 1 teaspoon coriander powder
- 1/2 teaspoon cayenne pepper (optional)
- Salt to taste

For the spice blend:
- 2 tablespoons coriander seeds
- 1 tablespoon cumin seeds
- 1 tablespoon cinnamon powder
- 1 tablespoon cardamom powder
- 1 tablespoon cloves powder
- 1 tablespoon saffron threads (optional)

**Instructions:**

1. **Prepare the marinade:** In a bowl, mix together yogurt, lemon juice, garam masala powder, cumin powder, coriander powder, cayenne pepper (if using), and salt. Add the chicken or beef and mix wel

### Compare different system roles



The same question can produce different styles when the system instruction changes.


In [ ]:
question = "Explain prompt engineering."

roles = [
    "You are a helpful teacher. Use simple language.",
    "You are a technical AI engineer. Give a precise technical explanation.",
    "You are a business trainer. Explain the concept using workplace examples.",
]

for role in roles:
    messages = [
        SystemMessage(content=role),
        HumanMessage(content=question),
    ]

    result = teacher_model.invoke(messages)

    print("=" * 70)
    print("SYSTEM ROLE:", role)
    print(result.content)
    print()

SYSTEM ROLE: You are a helpful teacher. Use simple language.
**What is Prompt Engineering?**

Prompt engineering is a technique used to create the best possible input, or "prompt," for a machine learning model. This is especially important for large language models like myself, which rely on the input we receive to generate accurate and helpful responses.

**Why is Prompt Engineering Important?**

Imagine you're asking me a question, but the way you phrase it is not clear. I might not understand what you're asking, and I might give you a confusing or incorrect answer. That's where prompt engineering comes in. By crafting a clear and specific prompt, you can help me understand what you're asking and give you a more accurate and helpful response.

**How Does Prompt Engineering Work?**

Here are some key steps to follow when using prompt engineering:

1. **Be clear and specific**: Make sure your prompt is easy to understand and clearly states what you're asking.
2. **Use the right languag

## 9. Multiple prompts

This example sends several independent questions using a loop.


In [ ]:
questions = [
    "What is Artificial Intelligence?",
    "What is Machine Learning?",
    "What is Generative AI?",
]

for question in questions:
    result = llm.invoke(question)

    print(f"Question: {question}")
    print(f"Answer: {result.content}")
    print("-" * 70)

Question: What is Artificial Intelligence?
Answer: Artificial Intelligence (AI) refers to the simulation of human intelligence in machines that are programmed to think and learn like humans. The term can also be applied to any machine that exhibits traits associated with a human mind such as learning and problem-solving.

AI technology is based on the principle of creating algorithms that can process data, identify patterns, and make decisions without being explicitly programmed for each specific task. This allows AI systems to adapt and improve over time, much like humans do.

There are several key characteristics of AI:

1. **Machine Learning**: AI systems can learn from data and improve their performance over time.
2. **Reasoning**: AI systems can draw conclusions and make decisions based on the data they have been trained on.
3. **Problem-Solving**: AI systems can identify and solve complex problems.
4. **Natural Language Processing**: AI systems can understand and generate human l

## 10. Streaming output

Streaming displays small parts of the response as soon as they are generated.


In [ ]:
stream_model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
)

for chunk in stream_model.stream(
    "Explain Large Language Models in simple words."
):
    print(chunk.content, end="", flush=True)

print()

**What are Large Language Models?**

Large Language Models (LLMs) are computer programs that can understand and generate human-like language. They're like super-smart language assistants that can read, write, and even talk like humans.

**How do they work?**

LLMs are trained on massive amounts of text data, which is like feeding them a huge library of books, articles, and conversations. This training helps them learn patterns, relationships, and structures of language.

Imagine you're teaching a child to speak by reading them books, having conversations, and showing them how words are used in different contexts. That's basically what LLMs do, but with much more data and complex algorithms.

**Key features of LLMs:**

1. **Understanding**: LLMs can comprehend the meaning of text, including nuances like sarcasm, idioms, and figurative language.
2. **Generation**: They can create new text based on what they've learned, like writing articles, stories, or even entire books.
3. **Conversati

## 11. Prompt Templates

A prompt template lets you reuse one prompt structure with different values.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert teacher for {domain}. "
            "Explain the answer in simple language using bullet points."
        ),
        (
            "human",
            "Explain {topic} and give one practical example."
        ),
    ]
)

formatted_messages = prompt_template.invoke(
    {
        "domain": "Generative AI",
        "topic": "Role in Health Sector",
    }
)

result = llm.invoke(formatted_messages)

print(result.content)

**Role of Generative AI in the Health Sector:**

* **Medical Imaging Analysis**: Generative AI can analyze medical images such as X-rays, CT scans, and MRIs to help doctors diagnose diseases more accurately and quickly.
* **Personalized Medicine**: Generative AI can help create personalized treatment plans for patients based on their genetic profiles, medical history, and lifestyle.
* **Predictive Analytics**: Generative AI can analyze large amounts of medical data to predict patient outcomes, identify high-risk patients, and prevent hospital readmissions.
* **Clinical Decision Support**: Generative AI can provide doctors with real-time clinical decision support, suggesting the best course of treatment based on the latest medical research and guidelines.
* **Virtual Nursing Assistants**: Generative AI can power virtual nursing assistants that can help patients with routine tasks, such as medication reminders and appointment scheduling.

**Practical Example:**

**Example:** A patient na

### Reuse the same template

Change only the variables instead of rewriting the complete prompt.


In [ ]:
topics = [
    ("Healthcare", "Generative AI for patient education"),
    ("Education", "AI-assisted lesson planning"),
    ("Business", "AI-generated meeting summaries"),
]

for domain, topic in topics:
    messages = prompt_template.invoke(
        {
            "domain": domain,
            "topic": topic,
        }
    )

    result = llm.invoke(messages)

    print("=" * 70)
    print(f"DOMAIN: {domain}")
    print(f"TOPIC: {topic}")
    print(result.content)
    print()

DOMAIN: Healthcare
TOPIC: Generative AI for patient education
**What is Generative AI?**

Generative AI is a type of artificial intelligence that can create new, original content, such as text, images, or videos, based on a given input or prompt. In the context of patient education, Generative AI can be used to create personalized and engaging educational materials for patients.

**Benefits of Generative AI for Patient Education:**

* **Personalization**: Generative AI can create content tailored to an individual patient's needs, health status, and language preferences.
* **Engagement**: AI-generated content can be more engaging and interactive, making it easier for patients to understand complex health information.
* **Scalability**: Generative AI can produce large volumes of content quickly and efficiently, making it ideal for large patient populations.
* **Consistency**: AI-generated content can ensure consistency in messaging and accuracy of information.

**Practical Example:**

**

## 12. Simple prompt-processing function

A function allows us to reuse the model with different questions.


In [ ]:
def ask_groq(question, temperature=0):
    model = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=temperature,
    )

    response = model.invoke(question)
    return response.content


answer = ask_groq(
    "Explain the difference between discriminative AI and generative AI."
)

print(answer)

Discriminative AI and generative AI are two fundamental approaches in the field of artificial intelligence (AI). The primary difference between them lies in their objectives and the types of tasks they are designed to perform.

**Discriminative AI:**

Discriminative AI models are designed to make predictions or classify data based on existing patterns and relationships. Their primary goal is to identify and distinguish between different classes or categories. These models typically use supervised learning techniques, where the model is trained on labeled data to learn the mapping between inputs and outputs.

Examples of discriminative AI tasks include:

1. Image classification: Identifying objects or scenes in images.
2. Speech recognition: Transcribing spoken words into text.
3. Sentiment analysis: Determining the emotional tone of text or speech.
4. Spam detection: Identifying spam emails or messages.

Discriminative AI models are typically trained to optimize a specific objective fu

# Part B — Hugging Face Embeddings

Groq is used for text generation in this notebook.

For embeddings, we use the free local Sentence Transformers model:

```text
sentence-transformers/all-MiniLM-L6-v2
```

The model is downloaded to the Colab runtime the first time this section runs.


## 13. Load the embedding model

The first execution can take some time because the model files must be downloaded.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Embedding model loaded successfully.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully.


## 14. Create an embedding for one query

An embedding converts text into a numerical vector.


In [ ]:
text = "Islamabad is the capital of Pakistan."

vector = embedding_model.embed_query(text)

print("Text:", text)
print("Vector length:", len(vector))
print("First 10 values:", vector[:10])

Text: Islamabad is the capital of Pakistan.
Vector length: 384
First 10 values: [0.018565168604254723, 0.08578694611787796, -0.05348091199994087, 0.09558005630970001, -0.0020012203603982925, -0.06132709980010986, 0.08399888873100281, -0.020370155572891235, -0.007159548811614513, 0.03674142062664032]


## 15. Create embeddings for multiple documents

Every document is converted into a vector of the same length.


In [ ]:
documents = [
    "Islamabad is the capital of Pakistan.",
    "Karachi is the largest city of Pakistan.",
    "Paris is the capital of France.",
]

document_vectors = embedding_model.embed_documents(documents)

print("Number of document vectors:", len(document_vectors))
print("Dimensions of each vector:", len(document_vectors[0]))
print("First 10 values of the first vector:")
print(document_vectors[1][:10])

Number of document vectors: 3
Dimensions of each vector: 384
First 10 values of the first vector:
[0.0509590283036232, 0.037472568452358246, -0.04346056282520294, 0.03756124526262283, -0.03180375322699547, -0.02301478385925293, 0.0423164963722229, -0.01980244554579258, -0.0589749701321125, 0.07337822765111923]


## 16. Embedding a Generative AI sentence


In [ ]:

sentence = "Generative AI can create new text, images, audio, and code."

sentence_vector = embedding_model.embed_query(sentence)

print("Sentence:", sentence)
print("Vector dimensions:", len(sentence_vector))

Sentence: Generative AI can create new text, images, audio, and code.
Vector dimensions: 384


## 17. Document similarity using cosine similarity

The query and documents are converted into embeddings. Cosine similarity measures how closely their meanings match.


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Virat Kohli is an Indian cricketer known for aggressive batting.",
    "MS Dhoni is a former Indian captain known for calm leadership.",
    "Sachin Tendulkar holds many international batting records.",
    "Rohit Sharma is known for elegant batting and double centuries.",
    "Jasprit Bumrah is an Indian fast bowler known for yorkers.",
]

query = "Tell me about calm Leadership."

document_vectors = embedding_model.embed_documents(documents)
query_vector = embedding_model.embed_query(query)

scores = cosine_similarity(
    [query_vector],
    document_vectors,
)[0]

best_index = int(np.argmax(scores))

print("Query:", query)
print()
print("Similarity scores:")

for document, score in zip(documents, scores):
    print(f"{score:.3f}  |  {document}")

print()
print("Best matching document:")
print(documents[best_index])
print("Similarity score:", round(float(scores[best_index]), 4))

Query: Tell me about calm Leadership.

Similarity scores:
0.204  |  Virat Kohli is an Indian cricketer known for aggressive batting.
0.480  |  MS Dhoni is a former Indian captain known for calm leadership.
0.012  |  Sachin Tendulkar holds many international batting records.
0.096  |  Rohit Sharma is known for elegant batting and double centuries.
0.018  |  Jasprit Bumrah is an Indian fast bowler known for yorkers.

Best matching document:
MS Dhoni is a former Indian captain known for calm leadership.
Similarity score: 0.4798


## 18. Display similarity results as a table


In [ ]:
import pandas as pd

results_df = pd.DataFrame(
    {
        "Document": documents,
        "Similarity Score": scores,
    }
).sort_values(
    by="Similarity Score",
    ascending=False,
)

results_df.reset_index(drop=True)

,Document,Similarity Score
0,MS Dhoni is a former Indian captain known for ...,0.479786
1,Virat Kohli is an Indian cricketer known for a...,0.204377
2,Rohit Sharma is known for elegant batting and ...,0.096386
3,Jasprit Bumrah is an Indian fast bowler known ...,0.017620
4,Sachin Tendulkar holds many international batt...,0.012181


## 19. Create a reusable semantic-search function


In [ ]:
def semantic_search(query, documents, top_k=3):
    document_vectors = embedding_model.embed_documents(documents)
    query_vector = embedding_model.embed_query(query)

    scores = cosine_similarity(
        [query_vector],
        document_vectors,
    )[0]

    ranked_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for index in ranked_indices:
        results.append(
            {
                "document": documents[index],
                "score": float(scores[index]),
            }
        )

    return results


search_results = semantic_search(
    query="Who is a fast bowler?",
    documents=documents,
    top_k=3,
)

for rank, item in enumerate(search_results, start=1):
    print(f"{rank}. Score: {item['score']:.4f}")
    print(item["document"])
    print()

1. Score: 0.5394
Jasprit Bumrah is an Indian fast bowler known for yorkers.

2. Score: 0.4869
Rohit Sharma is known for elegant batting and double centuries.

3. Score: 0.4712
Sachin Tendulkar holds many international batting records.



# Part C — Combined Mini Practical

In this activity:

1. Embeddings find the document most related to the question.
2. The retrieved document is sent to Groq.
3. Groq generates an answer using only the supplied context.

This is a simple introduction to the logic behind **Retrieval-Augmented Generation (RAG)**.


In [ ]:
knowledge_base = [
    "Islamabad became the capital of Pakistan in the 1960s.",
    "Karachi is Pakistan's largest city and a major commercial centre.",
    "Lahore is known for its history, culture, food, and educational institutions.",
    "Peshawar is the capital city of Khyber Pakhtunkhwa.",
]

user_question = "Which city is the capital of Sindh?"

retrieved_items = semantic_search(
    query=user_question,
    documents=knowledge_base,
    top_k=1,
)

retrieved_context = retrieved_items[0]["document"]

rag_messages = [
    SystemMessage(
        content=(
            "Answer the question using only the supplied context. "
            "If the answer is not in the context, say that the context "
            "does not contain enough information."
        )
    ),
    HumanMessage(
        content=(
            f"Context:\n{retrieved_context}\n\n"
            f"Question:\n{user_question}"
        )
    ),
]

response = llm.invoke(rag_messages)

print("Retrieved context:")
print(retrieved_context)
print()
print("Generated answer:")
print(response.content)

Retrieved context:
Karachi is Pakistan's largest city and a major commercial centre.

Generated answer:
The context does not contain enough information to determine which city is the capital of Sindh. However, it does mention that Karachi is Pakistan's largest city and a major commercial centre, which implies that Karachi is located in Sindh.


# Part D — Student Practice Tasks

## Task 1: Basic prompt

Ask Groq to explain one topic from your own professional domain.

## Task 2: Temperature

Run the same creative prompt at temperatures `0`, `0.7`, and `1.5`. Write down the differences.

## Task 3: System message

Create a system role for one of these professionals:

- Teacher
- Doctor
- Business manager
- Researcher
- Marketing specialist
- Software developer

## Task 4: Prompt template

Create a reusable template with variables for:

- Domain
- Topic
- Audience
- Output format

## Task 5: Embeddings

Create five short documents from your field and find the document most similar to a natural-language query.

## Task 6: Mini RAG

Retrieve the best matching document and ask Groq to answer using only that context.


# Important Notes

- Never share your Groq API key in notebook text or screenshots.
- Colab runtimes are temporary, so installed packages and downloaded models may disappear after the session ends.
- Generative AI outputs should be checked for accuracy, bias, and privacy risks.
- Human review is essential for healthcare, legal, financial, academic, and other high-impact uses.


# References

- Groq supported models: https://console.groq.com/docs/models
- LangChain ChatGroq integration: https://docs.langchain.com/oss/python/integrations/chat/groq
- LangChain Hugging Face integration: https://docs.langchain.com/oss/python/integrations/providers/huggingface
